# 数据探查：品牌多维度评分\n\n本 Notebook 用于完成样例数据的结构探查、质量检查，并形成“指标可支撑性”结论。

In [ ]:
from pathlib import Path\nimport json\nimport numpy as np\nimport pandas as pd\n\nROOT = Path('..').resolve()\nDATA_DIR = ROOT / 'sample-data'\n\ndef load_json(name):\n    with open(DATA_DIR / name, 'r', encoding='utf-8') as f:\n        return json.load(f)\n\nbrand = load_json('brand-info.json')\ncompany = load_json('company-info.json')\nproducts = load_json('products-info.json')\ntraffic = load_json('traffic-info.json')\nreviews = load_json('review-info.json')\n\nprint('Loaded files successfully.')

In [ ]:
# 1) 数据规模与结构概览\noverview = {\n    'brand_top_keys': list(brand.keys()),\n    'brand_domain_keys_count': len(brand.get('domain', {})),\n    'company_top_keys': list(company.keys()),\n    'company_cards_keys_count': len(company.get('cards', {})),\n    'products_count': len(products.get('products', [])),\n    'traffic_top_keys': list(traffic.keys()),\n    'reviews_count': len(reviews),\n}\noverview

In [ ]:
# 2) 核心字段抽样\ndomain = brand.get('domain', {})\nsample_brand = {\n    'merchant_name': domain.get('merchant_name'),\n    'created_at': domain.get('created_at'),\n    'estimated_visits': domain.get('estimated_visits'),\n    'estimated_sales_yearly': domain.get('estimated_sales_yearly'),\n    'trustpilot_review_count': domain.get('trustpilot', {}).get('review_count'),\n    'trustpilot_avg_rating': domain.get('trustpilot', {}).get('avg_rating'),\n    'product_count': domain.get('product_count'),\n}\nsample_brand

In [ ]:
# 3) 评价数据基础统计\nreview_df = pd.DataFrame(reviews)\nreview_df['stars'] = pd.to_numeric(review_df['stars'], errors='coerce')\nreview_df['text'] = review_df['bodyPositive'].fillna('') + ' ' + review_df['bodyNegative'].fillna('')\n\nreview_stats = {\n    'n_reviews': int(review_df.shape[0]),\n    'avg_stars': float(round(review_df['stars'].mean(), 3)),\n    'star_std': float(round(review_df['stars'].std(ddof=0), 3)),\n    'pct_low_star_<=2': float(round((review_df['stars'] <= 2).mean(), 3)),\n    'pct_high_star_>=4': float(round((review_df['stars'] >= 4).mean(), 3)),\n}\nreview_stats

In [ ]:
# 4) 文本关键词信号（质量/供应链/价格）\ntext = review_df['text'].str.lower()\n\nquality_neg_kw = ['artificial', 'aftertaste', 'too sweet', 'didn\'t taste good', 'not good']\nsupply_kw = ['shipping', 'delay', 'late', 'out of stock', 'delivery']\nvalue_pos_kw = ['worth', 'sale', 'deal', 'good alternative', 'better-for-you']\nvalue_neg_kw = ['expensive', 'pricey', 'cost a lot', 'too expensive']\n\ndef kw_ratio(series, keywords):\n    if len(series) == 0:\n        return 0.0\n    mask = pd.Series(False, index=series.index)\n    for k in keywords:\n        mask = mask | series.str.contains(k, regex=False)\n    return float(mask.mean())\n\nkw_stats = {\n    'quality_negative_ratio': round(kw_ratio(text, quality_neg_kw), 3),\n    'supply_issue_ratio': round(kw_ratio(text, supply_kw), 3),\n    'value_positive_ratio': round(kw_ratio(text, value_pos_kw), 3),\n    'value_negative_ratio': round(kw_ratio(text, value_neg_kw), 3),\n}\nkw_stats

In [ ]:
# 5) 产品数据基础统计\nprod_df = pd.DataFrame(products.get('products', []))\nprod_df['created_at'] = pd.to_datetime(prod_df['created_at'], errors='coerce')\nprod_df['published_at'] = pd.to_datetime(prod_df['published_at'], errors='coerce')\n\nall_variant_prices = []\navailable_variant_flags = []\nfor p in products.get('products', []):\n    for v in p.get('variants', []):\n        try:\n            all_variant_prices.append(float(v.get('price')))\n        except Exception:\n            pass\n        available_variant_flags.append(bool(v.get('available')))\n\nlatest_date = prod_df['created_at'].max()\nnew_365 = int((prod_df['created_at'] >= (latest_date - pd.Timedelta(days=365))).sum()) if pd.notna(latest_date) else 0\n\nprod_stats = {\n    'n_products': int(prod_df.shape[0]),\n    'n_variants_total': int(len(available_variant_flags)),\n    'available_variant_ratio': float(round(np.mean(available_variant_flags) if available_variant_flags else 0.0, 3)),\n    'price_mean': float(round(np.mean(all_variant_prices), 2)) if all_variant_prices else None,\n    'price_min': float(round(np.min(all_variant_prices), 2)) if all_variant_prices else None,\n    'price_max': float(round(np.max(all_variant_prices), 2)) if all_variant_prices else None,\n    'new_products_last_365d': new_365,\n}\nprod_stats

In [ ]:
# 6) 流量数据统计\nmonthly_visits = traffic.get('EstimatedMonthlyVisits', {})\nmv = pd.Series(monthly_visits, dtype='float64')\nmv.index = pd.to_datetime(mv.index)\nmv = mv.sort_index()\n\ntraffic_stats = {\n    'last_3m_visits': mv.to_dict(),\n    'visits_mean_3m': float(round(mv.mean(), 2)) if not mv.empty else None,\n    'visits_cv_3m': float(round((mv.std(ddof=0) / mv.mean()), 3)) if (not mv.empty and mv.mean() > 0) else None,\n    'global_rank': traffic.get('GlobalRank', {}).get('Rank'),\n    'category_rank': traffic.get('CategoryRank', {}).get('Rank'),\n}\ntraffic_stats

## 维度可支撑性判断（初稿）\n\n- 品牌成熟度：高可支撑（品牌成立时间、流量、排名、渠道信息充分）\n- 产品质量：中高可支撑（星级+评论文本；样本量中等）\n- 市场需求匹配度：中可支撑（流量与评论热度可代理需求，缺销量明细）\n- 创新力：中可支撑（新品发布时间与公司动态可用，专利数据缺失）\n- 供应链可靠性：低到中可支撑（评论中供应链关键词较少，信号弱）\n- 性价比：中可支撑（价格分布+价格感知评论可用）\n- 可持续发展：低可支撑（缺正式 ESG/认证结构化字段）\n- 市场趋势契合度：中可支撑（可用关键词近似，但缺外部趋势基准）

## 指标设计结论\n\n本项目优先启用 6 个维度进入评分：\n\n1. 品牌成熟度\n2. 产品质量\n3. 市场需求匹配度\n4. 创新力\n5. 供应链可靠性（低置信度）\n6. 性价比\n\n并在正式指标设计文档中给出每个维度的公式、权重与置信度处理策略。